# 05 · Poda → queda: o chamado de poda antecipa a queda da árvore?

[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/renanbrettas/Projeto-ci-ncia-de-dados---Podas-de-rvore/blob/main/05_poda_queda.ipynb)

**Projeto:** Chamados de poda de árvore no Rio de Janeiro &nbsp;·&nbsp; **Parte 5 de 5**

---

### Contexto
Este é o núcleo da pergunta para a prefeitura: se um pedido de poda sinaliza risco, deveria haver **mais quedas de árvore depois dele** no mesmo local do que o acaso explicaria.

### Pergunta
> Depois de um chamado de poda num logradouro, a chance de árvore caída ali é maior que o esperado?

### Dados utilizados
- `data/poda_2021_2025.parquet` (notebook 03)
- `adm_central_atendimento_1746.chamado` (quedas)

### Roteiro
1. Eventos índice (podas) e desfechos (quedas) no mesmo logradouro
2. Incidência em 7, 30, 90, 180 e 365 dias vs. linha de base de Poisson
3. Diagnóstico de direção (quedas *antes* da poda)
4. Tempo até a primeira queda

### Principais resultados
- A partir de 30 dias, a queda depois da poda é **menos** frequente que o acaso (razão obs/esp de 0,74 a 0,85).
- O excesso em 7 dias aparece **nas duas direções** (1,15 depois, 1,39 antes): sinal de registro conjunto do mesmo episódio.
- Conclusão: o volume de chamados de poda, sozinho e por logradouro, **não antecipa quedas**.

> ℹ️ As saídas deste notebook são as da execução original no Google Colab (dados extraídos do BigQuery
> em novembro de 2026), então os resultados podem ser lidos sem rodar nada. Para reproduzir, rode a
> célula de configuração abaixo (veja o [README](README.md)).

### Desenho do estudo
- **Eventos índice:** 131.874 chamados de poda (2021–2025) com logradouro válido.
- **Desfecho:** chamados de `Retirada de árvore caída` desde 2021 no mesmo logradouro.
- **Horizontes:** 7, 30, 90, 180 e 365 dias, contando só podas com seguimento completo.
- **Linha de base:** Poisson por logradouro, P(≥1 queda em H dias) = 1 − exp(−λ·H), com λ = quedas do logradouro ÷ duração do período.
- **Diagnóstico de direção:** a mesma conta para quedas **antes** da poda.

In [ ]:
# Configuração (rode primeiro)
import os
import subprocess
import sys
from pathlib import Path

# No Colab, informe o projeto do Google Cloud que vai pagar a leitura (veja o README):
# os.environ["GCP_PROJECT_ID"] = "seu-projeto-gcp"

RAIZ = Path.cwd()
if not (RAIZ / "src" / "podas").exists():  # aberto direto no Colab: baixa o código do repositório
    RAIZ = Path("/content/analise-podas")
    if not RAIZ.exists():
        subprocess.run(["git", "clone", "-q", "https://github.com/renanbrettas/Projeto-ci-ncia-de-dados---Podas-de-rvore.git", str(RAIZ)], check=True)
sys.path.insert(0, str(RAIZ / "src"))

import pandas as pd
from google.cloud import bigquery

from podas.bq import get_client, rodar
from podas.config import ALVO, SENSIBILIDADE, ARQ_PODA, DATA_DIR, FIG_DIR, OUT_DIR

client = get_client()

import numpy as np

In [ ]:
poda_df = pd.read_parquet(ARQ_PODA)  # extração do notebook 03

In [ ]:
import numpy as np
from datetime import datetime

SUBTIPOS_QUEDA = ["Retirada de árvore caída"]   # AJUSTAR depois do passo 1
HORIZ = [7, 30, 90, 180, 365]                   # dias
INI = pd.Timestamp("2021-01-01")

# 1) Eventos índice: poda com logradouro válido
inv = poda_df["id_logradouro"].isna() | poda_df["id_logradouro"].astype(str).str.strip().isin(["", "0"])
idx = poda_df.loc[~inv, ["id_chamado", "id_logradouro", "data_inicio"]].copy()
idx["id_logradouro"] = idx["id_logradouro"].astype(str).str.strip()
idx["t0"] = pd.to_datetime(idx["data_inicio"])
print(len(idx), "chamados de poda (esperado: 131874)")

# 2) Desfechos: queda desde 2021, sem teto de data (inclui 2026 como seguimento)
Pq = [bigquery.ArrayQueryParameter("subs", "STRING", SUBTIPOS_QUEDA),
      bigquery.ScalarQueryParameter("ini", "DATETIME", datetime(2021, 1, 1))]
q = rodar("""
SELECT id_chamado, id_logradouro, data_inicio, subtipo
FROM `datario.adm_central_atendimento_1746.chamado`
WHERE subtipo IN UNNEST(@subs) AND data_inicio >= @ini
  AND id_logradouro IS NOT NULL AND TRIM(id_logradouro) NOT IN ('', '0')
""", Pq)
q["id_logradouro"] = q["id_logradouro"].astype(str).str.strip()
q["t1"] = pd.to_datetime(q["data_inicio"])
print(len(q), "chamados de queda com logradouro válido:", q["subtipo"].value_counts().to_dict())

T_FIM = pd.to_datetime(rodar(
    "SELECT MAX(data_inicio) AS d FROM `datario.adm_central_atendimento_1746.chamado`")["d"].iloc[0])
print("última data_inicio na tabela:", T_FIM)

# 3) Pares poda × queda no mesmo logradouro
pares = idx.merge(q[["id_logradouro", "t1"]], on="id_logradouro", how="inner")
pares["delta"] = (pares["t1"] - pares["t0"]).dt.total_seconds() / 86400   # > 0: queda depois da poda

dep = pares[pares["delta"] > 0].groupby("id_chamado")["delta"].min()
ant = (-pares[pares["delta"] < 0].set_index("id_chamado")["delta"]).groupby(level=0).min()
idx["dias_ate_queda"] = idx["id_chamado"].map(dep)        # NaN = nenhuma queda depois
idx["dias_desde_queda"] = idx["id_chamado"].map(ant)      # NaN = nenhuma queda antes

idx["seguimento"] = (T_FIM - idx["t0"]).dt.total_seconds() / 86400
idx["historico"] = (idx["t0"] - INI).dt.total_seconds() / 86400
span = (T_FIM - INI).total_seconds() / 86400
idx["lam"] = idx["id_logradouro"].map(q.groupby("id_logradouro").size() / span).fillna(0.0)

# 4) Quantos (incidência acumulada), contra o acaso e contra a direção inversa
def inc(col_dias, col_cob, H):
    d = idx[idx[col_cob] >= H]
    obs = int((d[col_dias] <= H).sum())
    esp = float((1 - np.exp(-d["lam"] * H)).sum())   # Poisson por logradouro, momento independente
    return {"horizonte_dias": H, "n_poda": len(d), "com_queda": obs,
            "pct_obs": round(100 * obs / len(d), 2),
            "pct_esperado_acaso": round(100 * esp / len(d), 2),
            "razao_obs_esp": round(obs / esp, 2) if esp > 0 else None}

depois = pd.DataFrame([inc("dias_ate_queda", "seguimento", H) for H in HORIZ])
antes = pd.DataFrame([inc("dias_desde_queda", "historico", H) for H in HORIZ])
print("\nQUEDA DEPOIS DA PODA\n", depois.to_string(index=False))
print("\nQUEDA ANTES DA PODA (diagnóstico de direção)\n", antes.to_string(index=False))

# 5) Em quanto tempo (só entre quem teve queda em até 365 dias)
d365 = idx[(idx["seguimento"] >= 365) & (idx["dias_ate_queda"] <= 365)]
print("\natraso até a primeira queda, n =", len(d365))
print(d365["dias_ate_queda"].describe(percentiles=[.25, .5, .75]).round(1))
print("queda a menos de 1 dia da poda:", int((idx["dias_ate_queda"] < 1).sum()))

131874 chamados de poda (esperado: 131874)
custo estimado: 1.383 GB
10585 chamados de queda com logradouro válido: {'Retirada de árvore caída': 10585}
custo estimado: 0.162 GB
última data_inicio na tabela: 2026-09-20 13:32:31

QUEDA DEPOIS DA PODA
  horizonte_dias  n_poda  com_queda  pct_obs  pct_esperado_acaso  razao_obs_esp
              7  131874       1559     1.18                1.03           1.15
             30  131874       4277     3.24                4.07           0.80
             90  131874      10111     7.67               10.35           0.74
            180  131874      17188    13.03               17.28           0.75
            365  126150      28686    22.74               26.79           0.85

QUEDA ANTES DA PODA (diagnóstico de direção)
  horizonte_dias  n_poda  com_queda  pct_obs  pct_esperado_acaso  razao_obs_esp
              7  131429       1887     1.44                1.03           1.39
             30  129793       4655     3.59                4.07         

## Leitura dos resultados

| Horizonte | Queda depois da poda | Esperado pelo acaso | Razão obs/esp (depois) | Razão obs/esp (antes) |
|---|---|---|---|---|
| 7 dias | 1,18% | 1,03% | **1,15** | **1,39** |
| 30 dias | 3,24% | 4,07% | 0,80 | 0,88 |
| 90 dias | 7,67% | 10,35% | 0,74 | 0,78 |
| 180 dias | 13,03% | 17,28% | 0,75 | 0,78 |
| 365 dias | 22,74% | 26,79% | 0,85 | 0,80 |

- **Não há sinal preditivo a partir de 30 dias:** a queda depois de uma poda é *menos* frequente
  do que o acaso prevê (razão entre 0,74 e 0,85).
- O pequeno excesso em 7 dias aparece **nas duas direções**, e é maior antes da poda (1,39) do que
  depois (1,15). Isso indica registro conjunto do mesmo episódio (por exemplo, poda e queda abertas
  juntas após uma tempestade), não uma poda que antecede a queda. 538 quedas ocorrem a menos de
  1 dia da poda.
- Entre quem teve queda em até 365 dias, a mediana é de 158 dias até a primeira queda.

**Ressalvas:** a unidade logradouro mistura vias muito longas; a linha de base supõe taxa constante
no tempo (não modela sazonalidade nem chuva); e o atendimento da poda (que pode *prevenir* quedas)
não é observado. O próximo passo é refazer a análise por **CEP / trecho** (`src/podas/cep.py`) e
incluir a **chuva** como covariável.

---

⬅️ [04 · Análise espacial: bairros e logradouros](04_analise_espacial.ipynb) &nbsp;|&nbsp; [🏠 Início](README.md)